## Validation of `client.py` Functionality:
1. **Configuration**: Connection parameters from `config.yaml`.
2. **Instantiation**: `CenaceClient` object initialization.
3. **HTTP Request**: 1-day download for 1 node (`06MON-115`) in the **MDA** market.
4. **JSON Inspection**: Verification of the raw response dictionary structure from CENACE.
5. **Cache Isolation**: Verification that different nodes do not overwrite cache files.

In [17]:
import sys
from pathlib import Path
from datetime import date
import yaml
import pandas as pd

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.extract.client import CenaceClient

print("Project root directory:", ROOT_DIR)


Project root directory: /home/jose/prueba/challenge


## Load Connection Parameters (`config.yaml`)

In [18]:
config_path = ROOT_DIR / "config.yaml"
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Electrical system:", config["system"])
print("Base URL:", config["api"]["base_url"])
print(f"Configured nodes ({len(config['nodes'])}):", config["nodes"])


Electrical system: SIN
Base URL: https://ws01.cenace.gob.mx:8082/SWPML/SIM
Configured nodes (9): ['06AEO-115', '06AHM-400', '06FRO-115', '06GRA-115', '06MON-115', '06MON-230', '06RGS-115', '06TKS-115', '06XOC-115']


In [19]:
raw_dir = ROOT_DIR / config["paths"]["raw_data_dir"]

client = CenaceClient(
    base_url=config["api"]["base_url"],
    system=config["system"],
    raw_data_dir=raw_dir,
    timeout_seconds=config["api"]["timeout_seconds"],
    max_retries=config["api"]["max_retries"],
    backoff_factor=config["api"]["backoff_factor"],
    rate_limit_delay_seconds=0.5,
)


## Live API Request (1 node, 1 day)
We execute a live query for node `06MON-115` on `2024-01-01`.

In [20]:
test_node = ["06MON-115"]
test_date = date(2024, 1, 1)

print(f"Querying CENACE for node {test_node} on date {test_date}...")
response = client.fetch_window(
    process="MDA",
    nodes=test_node,
    start=test_date,
    end=test_date,
    force_reload=True,
)

print("\n--- Server Response ---")
print("Status:", response.get("status"))
print("System:", response.get("sistema"))
print("Process:", response.get("proceso"))
print("Available keys in JSON:", list(response.keys()))


Querying CENACE for node ['06MON-115'] on date 2024-01-01...

--- Server Response ---
Status: OK
System: SIN
Process: MDA
Available keys in JSON: ['nombre', 'proceso', 'sistema', 'area', 'Resultados', 'status']


## Inspect JSON Structure

In [21]:
resultados = response.get("Resultados", [])
print(f"Total returned nodes: {len(resultados)}")

if resultados:
    nodo_info = resultados[0]
    print("Reported node:", nodo_info.get("clv_nodo"))
    valores = nodo_info.get("Valores", [])
    print(f"Returned hours: {len(valores)}")
    print("\nSample of the first 3 hours:")
    for v in valores[:3]:
        print(" ", v)
else:
    print("No results found in payload.")


Total returned nodes: 1
Reported node: 06MON-115
Returned hours: 24

Sample of the first 3 hours:
  {'fecha': '2024-01-01', 'hora': '1', 'pml': '279.54', 'pml_ene': '370.71', 'pml_per': '-44.07', 'pml_cng': '-47.09'}
  {'fecha': '2024-01-01', 'hora': '2', 'pml': '278.89', 'pml_ene': '311.75', 'pml_per': '-32.86', 'pml_cng': '0'}
  {'fecha': '2024-01-01', 'hora': '3', 'pml': '273.94', 'pml_ene': '305.52', 'pml_per': '-31.57', 'pml_cng': '0'}


## Cache Isolation for Multiple Nodes
Verify that querying multiple nodes produces a distinct filename to prevent data collisions.

In [22]:
cache_path = client._get_cache_path("MDA", test_node, test_date, test_date)
dos_nodos = ["06AEO-115", "06MON-115"]
ruta_multi = client._get_cache_path("MDA", dos_nodos, test_date, test_date)

print("Single-node file: ", cache_path.name)
print("Multi-node file:  ", ruta_multi.name)
assert cache_path.name != ruta_multi.name, "Error: cache paths collide!"
print("\nVerification successful: cache paths are isolated.")


Single-node file:  sin_mda_06MON-115_2024-01-01_2024-01-01.json
Multi-node file:   sin_mda_2n_687aea39_2024-01-01_2024-01-01.json

Verification successful: cache paths are isolated.


## Test `windows.py` (7-Day Date Window Slicing)
Test the `generate_date_windows` function across the date range defined in `config.yaml`.

In [23]:
from src.extract.windows import generate_date_windows

start_dt = date.fromisoformat(config["date_range"]["start"])
end_dt = date.fromisoformat(config["date_range"]["end"])
windows = generate_date_windows(start_dt, end_dt, max_days=config["api"]["max_days_per_request"])

print(f"Total date windows generated: {len(windows)}")
print(f"First window: {windows[0][0]} -> {windows[0][1]}")
print(f"Last window:  {windows[-1][0]} -> {windows[-1][1]}")


Total date windows generated: 79
First window: 2024-01-01 -> 2024-01-07
Last window:  2025-06-30 -> 2025-06-30


## Test `parse.py` (JSON to DataFrame Transformation)
Parse the cached JSON on disk and inspect the resulting DataFrame and data types.

In [24]:
from src.extract.parse import parse_raw_jsons_to_dataframe

df = parse_raw_jsons_to_dataframe(raw_dir=raw_dir, process="MDA")
print(f"Table dimensions: {df.shape}")
print("\nColumn data types:")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())


Table dimensions: (118152, 9)

Column data types:
clv_nodo             object
timestamp    datetime64[ns]
fecha                object
hora                  int64
pml                 float64
pml_ene             float64
pml_per             float64
pml_cng             float64
proceso              object
dtype: object

First 5 rows:
    clv_nodo           timestamp       fecha  hora     pml  pml_ene  pml_per  \
0  06AEO-115 2024-01-01 00:00:00  2024-01-01     1  279.92   370.71   -43.69   
1  06AEO-115 2024-01-01 01:00:00  2024-01-01     2  279.21   311.75   -32.54   
2  06AEO-115 2024-01-01 02:00:00  2024-01-01     3  274.23   305.52   -31.28   
3  06AEO-115 2024-01-01 03:00:00  2024-01-01     4  264.01   294.23   -30.22   
4  06AEO-115 2024-01-01 04:00:00  2024-01-01     5  243.44   267.66   -24.22   

   pml_cng proceso  
0    -47.1     MDA  
1      0.0     MDA  
2      0.0     MDA  
3      0.0     MDA  
4      0.0     MDA  


## Structural Inspection of the Full Dataset (`pml_mda.parquet`)
Verify data integrity of the complete Parquet dataset covering all 547 days and 9 Monclova nodes.

In [25]:
import pandas as pd

parquet_file = ROOT_DIR / "data/interim/pml_mda.parquet"
df_full = pd.read_parquet(parquet_file)

print(f"Total records: {len(df_full):,}")
print(f"Temporal range: {df_full['timestamp'].min()} to {df_full['timestamp'].max()}")
print(f"Unique nodes ({df_full['clv_nodo'].nunique()}): {sorted(df_full['clv_nodo'].unique())}")
print("\nMissing values per column:")
print(df_full.isnull().sum())

print("\nHour counts per node (Expected: 13,128 per node):")
counts = df_full["clv_nodo"].value_counts()
print(counts)

# Integrity assertions
assert len(df_full) == 118152, f"Expected 118,152 rows, found {len(df_full)}"
assert df_full["clv_nodo"].nunique() == 9, "Missing nodes in dataset"
assert df_full.isnull().sum().sum() == 0, "Missing values found in dataset"
assert counts.min() == 13128 and counts.max() == 13128, "Nodes found with incomplete hours"

print("\nStructural verification completed successfully: dataset is 100% complete and intact!")


Total records: 118,152
Temporal range: 2024-01-01 00:00:00 to 2025-06-30 23:00:00
Unique nodes (9): ['06AEO-115', '06AHM-400', '06FRO-115', '06GRA-115', '06MON-115', '06MON-230', '06RGS-115', '06TKS-115', '06XOC-115']

Missing values per column:
clv_nodo     0
timestamp    0
fecha        0
hora         0
pml          0
pml_ene      0
pml_per      0
pml_cng      0
proceso      0
dtype: int64

Hour counts per node (Expected: 13,128 per node):
clv_nodo
06AEO-115    13128
06AHM-400    13128
06FRO-115    13128
06GRA-115    13128
06MON-115    13128
06MON-230    13128
06RGS-115    13128
06TKS-115    13128
06XOC-115    13128
Name: count, dtype: int64

Structural verification completed successfully: dataset is 100% complete and intact!


In [29]:
display(df_full.head())
display(df_full.describe())

,clv_nodo,timestamp,fecha,hora,pml,pml_ene,pml_per,pml_cng,proceso
0,06AEO-115,2024-01-01 00:00:00,2024-01-01,1,279.92,370.71,-43.69,-47.1,MDA
1,06AEO-115,2024-01-01 01:00:00,2024-01-01,2,279.21,311.75,-32.54,0.0,MDA
2,06AEO-115,2024-01-01 02:00:00,2024-01-01,3,274.23,305.52,-31.28,0.0,MDA
3,06AEO-115,2024-01-01 03:00:00,2024-01-01,4,264.01,294.23,-30.22,0.0,MDA
4,06AEO-115,2024-01-01 04:00:00,2024-01-01,5,243.44,267.66,-24.22,0.0,MDA


,timestamp,hora,pml,pml_ene,pml_per,pml_cng
count,118152,118152.000000,118152.000000,118152.000000,118152.000000,118152.000000
mean,2024-09-30 11:30:00,12.500000,784.435594,844.497173,-48.770860,-11.291345
min,2024-01-01 00:00:00,1.000000,-53.870000,-0.010000,-603.850000,-2612.110000
25%,2024-05-16 17:45:00,6.750000,387.497500,421.010000,-64.010000,-25.220000
50%,2024-09-30 11:30:00,12.500000,534.405000,577.570000,-39.790000,-3.120000
75%,2025-02-14 05:15:00,18.250000,775.472500,867.355000,-22.310000,19.500000
max,2025-06-30 23:00:00,24.000000,26222.270000,25590.570000,805.440000,3717.140000
std,NaN,6.922216,1060.391936,1069.081389,49.487748,180.437936
